# 단어 임베딩 행렬 생성

In [2]:
# 실습용 데이터 생성

"""
1. 입력 text: "I love you"
2. 토큰화 -> 크기가 30000인 단어 사전에 근거(단어 사전 상의 토큰) -> [I, love, you]
3. 단어 사전상의 토큰별 위치(인덱스)를 이용 -> 정수 인덱스로 변환 -> [250, 12000, 700]
4. Embedding Layer -> 2차원 배열(행: 단어 사전의 크기, 열: 512) -> (3, 512)모양의 임베딩 행렬
5. (3, 512)모양의 임베딩 행렬 + 위치 정보를 표현하는 (3, 512) 배열 덧셈 -> 각 단어 별 임베딩 백터(3, 512)
6. 실습: (3, 512) 모양의 2차원 배열 생성 -> 표준 정규 분포로분터 랜덤한 실수 샘플링
"""

# 필요한 라이브러리 임포트
import torch

# (3, 512) 임베딩 행렬에 해당하는 데이터 생성
d_model=512
embedding_matrix_shape = (3, d_model)
embedding_matrix = torch.randn(embedding_matrix_shape)

# 결과 확인
print(embedding_matrix)

tensor([[-0.1291,  0.1361, -0.6687,  ...,  0.5382,  0.3761,  1.3776],
        [-0.2689, -0.4662,  0.9727,  ...,  0.2898, -1.5331, -0.7307],
        [ 1.1112, -0.4342, -2.1073,  ..., -0.7945, -1.5240, -0.5901]])


# 가중치 행렬 생성

In [3]:
# 가중치 W_q, W_k, W_v 생성

"""
1. 각 각중치 행렬 W_q, W_k, W_v의 모양: (512, 512)
2. 각 가중치 행렬 개별적으로 생성(X), 한 번에 묶어서 생성
3. 전체 -> 가중치 행렬 1개 -> 모양 (512, 512*3)
4. 표준 정규 분포로부터 랜덤한 실수 샘플링 -> torch.randn() 함수
"""
# 필요한 모듈 임포트
import math

# torch.randn() 함수 호출, 가중치 행렬 생성 (512, 512*3) 및 scaling
weights_matrix=torch.randn(d_model, d_model*3) / math.sqrt(d_model)

# 결과 확인
print(weights_matrix)

tensor([[ 0.0443, -0.0304,  0.0177,  ..., -0.0651, -0.0233, -0.0177],
        [-0.0202,  0.0136,  0.0779,  ...,  0.0139, -0.0005,  0.0058],
        [-0.0162, -0.0163,  0.0430,  ...,  0.0602,  0.0061, -0.0263],
        ...,
        [-0.0467,  0.0191, -0.0457,  ..., -0.0221,  0.0225,  0.0472],
        [-0.0338, -0.0638, -0.0041,  ...,  0.0434,  0.0563,  0.0132],
        [-0.0387, -0.0451, -0.0241,  ...,  0.0962,  0.0130, -0.0404]])


In [4]:
# 가중치 행렬: W_q, W_k, W_v -> 2차원 배열의 슬라이싱
W_q = weights_matrix[:, 0:512]
W_k = weights_matrix[:, 512:1024]
W_v = weights_matrix[:, 1024:]

# 결과 확인
print(W_q)
print(f'W_q 모양: {W_q.shape}')
print('-'*80)
print(W_k)
print(f'W_k 모양: {W_k.shape}')
print('-'*80)
print(W_v)
print(f'W_v 모양: {W_v.shape}')

tensor([[ 0.0443, -0.0304,  0.0177,  ...,  0.0342,  0.0041, -0.0204],
        [-0.0202,  0.0136,  0.0779,  ..., -0.0053,  0.0512,  0.0144],
        [-0.0162, -0.0163,  0.0430,  ...,  0.0133,  0.0038, -0.0126],
        ...,
        [-0.0467,  0.0191, -0.0457,  ..., -0.0379, -0.0197, -0.0232],
        [-0.0338, -0.0638, -0.0041,  ...,  0.0101, -0.0259,  0.0366],
        [-0.0387, -0.0451, -0.0241,  ..., -0.0660,  0.0317, -0.0699]])
W_q 모양: torch.Size([512, 512])
--------------------------------------------------------------------------------
tensor([[-0.0327, -0.0181, -0.0545,  ..., -0.0073, -0.0331,  0.0594],
        [ 0.0065,  0.0113, -0.0118,  ..., -0.1122, -0.0402, -0.0182],
        [-0.0169, -0.0392, -0.0170,  ...,  0.0218, -0.0036, -0.0037],
        ...,
        [ 0.0251, -0.0084,  0.0230,  ...,  0.0181,  0.0466,  0.0504],
        [-0.0260,  0.0235,  0.0122,  ..., -0.0277, -0.0277, -0.0590],
        [-0.0350, -0.0389, -0.0445,  ..., -0.0495, -0.0483, -0.0348]])
W_k 모양: torch.Size([

# q, k, v 행렬 생성

In [5]:
# q, k, v 생성

"""
# 단어의 임베딩 행렬과 가중치 행렬의 행렬 곱 -> 각 단어별 q, k, v 행렬 생성
"""

# 전체 단어의 임베딩 행렬과 가중치 행렬의 곱 -> matmul() 함수

qkv = torch.matmul(embedding_matrix, weights_matrix)
print(qkv)
print(f'qkv 행렬의 모양: {qkv.shape}')

print('-'*80)

# 각 단어의 q, k, v  행렬 추출
query = qkv[:, 0:512]
key = qkv[:, 512:1024]
value = qkv[:, 1024:]

# 결과 확인
print(f"query 행렬 모양: {query.shape}")
print('-'*80)
print(f'key 행렬 모양: {key.shape}')
print('-'*80)
print(f'value 행렬 모양: {value.shape}')

tensor([[-0.3513,  1.6575, -0.8004,  ...,  0.3620,  0.1203, -0.2885],
        [ 0.5064, -0.2749,  0.2125,  ..., -0.1393,  0.2291, -0.1238],
        [ 1.4867,  0.7135,  0.8162,  ...,  0.6409, -1.8825, -1.3017]])
qkv 행렬의 모양: torch.Size([3, 1536])
--------------------------------------------------------------------------------
query 행렬 모양: torch.Size([3, 512])
--------------------------------------------------------------------------------
key 행렬 모양: torch.Size([3, 512])
--------------------------------------------------------------------------------
value 행렬 모양: torch.Size([3, 512])


# scaled dot product attention

## attention score: query 행렬과 key 행렬 간의 곱셈

In [11]:
# query 행렬 (3, 512), key 행렬 (3, 512)의 곱셉

## 행렬 간 곱셈의 결과 값이 클수록 해당 단어 간의 연관성이 크다 -> 점수: score
## 핵심: key 행렬의 모양 -> Transpose(matrix Transpose) -> (512, 3)
attention_score = torch.matmul(query, key.mT)
print(f'attention_score: \n{attention_score}')

attention_score: 
tensor([[ 13.2806, -20.6805,  -5.2088],
        [ 23.6074,  12.8819,  22.7510],
        [ 22.6296,  -5.7053, -40.8871]])


## scaling

In [13]:
# attention_score -> 1보다 큰 숫자를 이용행서 나눗셈 -> 숫자 크기 감소
scaled_attention_score = attention_score / math.sqrt(d_model)
print(f'scaled_attention_score: \n{scaled_attention_score}')

scaled_attention_score: 
tensor([[ 0.5869, -0.9140, -0.2302],
        [ 1.0433,  0.5693,  1.0055],
        [ 1.0001, -0.2521, -1.8070]])


## attention weight

In [14]:
# 필요한 함수 임포트 (신경망과 관계없이 단순한 연산이 필요)
import torch.nn.functional as F

# scaling 한 attention score -> softmax 함수에 입력
attention_weight = F.softmax(scaled_attention_score)
print(f'attention_weight: \n{attention_weight}')

attention_weight: 
tensor([[0.6007, 0.1339, 0.2653],
        [0.3868, 0.2408, 0.3724],
        [0.7428, 0.2123, 0.0449]])


/tmp/ipykernel_584/4272510922.py:5: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attention_weight = F.softmax(scaled_attention_score)


## attention output: contextual embedding vector

In [15]:
# 반영 비율(attention weight) x (각 단어의 임베딩 행렬)의 결과
'''
# value != output(각 단어에 대한 반영 비율이 포함되서 계산된 결과)
# output: 단어 1개에 대한 임베딩 백터가 아니라 문장의 모든 단어들간의
  관계를 반영한 임베딩 백터
# 예시: 새로운 I의 임베딩 백터 =
        [1번 단어 I 정보 x 90%] + [2번 단어 love 정보 x 3%] + [3번 단어 정보 you x 7%]

'''
output = torch.matmul(attention_weight, value)
print(f'output: \n{output}')

output: 
tensor([[-0.1524,  0.1041,  1.2003,  ...,  0.3689, -0.3966, -0.5353],
        [-0.3224,  0.1818,  0.9386,  ...,  0.3452, -0.5994, -0.6262],
        [ 0.1384, -0.2345,  1.0620,  ...,  0.2681,  0.0536, -0.2990]])


##